# II-02. 곡선 맞춤과 모형 평가

「지구과학 데이터 사이언스」 책의 코드입니다. 설명은 책을 참고하세요.

**준비** (책에는 싣지 않은 공통 셀)

In [ ]:
# 필요한 모듈 정의 (버전 명시 가능)
packages_str = "numpy, matplotlib, scipy, version_information"

In [ ]:
# 공통 함수(dses_utils) 불러오기 — 코랩/로컬 모두 동작(별도 설치 불필요)
import os, sys, importlib, urllib.request

_url = "https://raw.githubusercontent.com/Kiehyun/dses_utils/main/dses_utils.py"
try:
    _new = urllib.request.urlopen(_url, timeout=10).read()   # 최신 내용을 메모리로 받아옴
    _old = open("dses_utils.py", "rb").read() if os.path.exists("dses_utils.py") else None
    if _new != _old:                                          # 내용이 다를 때만 덮어씀
        with open("dses_utils.py", "wb") as _f:
            _f.write(_new)
        if "dses_utils" in sys.modules:                       # 이미 불러왔다면 다시 불러옴
            importlib.reload(sys.modules["dses_utils"])
except Exception:
    pass  # 인터넷이 안 되면 기존에 받아둔 dses_utils.py 사용
import dses_utils as du
du.check_and_install(packages_str)
du.set_korean_font()  # 그래프의 한글 글꼴 설정
proj_dir = du.make_project_dir(du.current_notebook_name())  # 데이터 저장 폴더

**code 25.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

TRUE_SLOPE, TRUE_INTERCEPT = 2.5, 1.0        # 참값(실제로는 모르는 값)
x = np.linspace(0, 10, 25)
y = TRUE_SLOPE * x + TRUE_INTERCEPT + rng.normal(0, 1.5, x.size)

print("데이터 점 개수:", x.size)
print("x 범위 :", x.min(), "~", x.max())
print("y 범위 :", round(y.min(), 2), "~", round(y.max(), 2))

**code 26.**

In [ ]:
slope, intercept = np.polyfit(x, y, 1)  # 1차 = 직선 맞춤
y_fit = slope * x + intercept           # 맞춘 직선의 예측값
residual = y - y_fit                    # 잔차 = 실제값 - 예측값

print(f"맞춘 기울기 : {slope:.3f}  (참값 {TRUE_SLOPE})")
print(f"맞춘 절편   : {intercept:.3f}  (참값 {TRUE_INTERCEPT})")
print(f"잔차 제곱합 : {np.sum(residual**2):.2f}")
print(f"잔차 평균   : {residual.mean():.3e}  (최소제곱이면 거의 0)")

**code 27.**

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.5, 5.5), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

axes[0].scatter(x, y, color="steelblue", label="관측값")
axes[0].plot(x, y_fit, "r-", lw=2,
             label=f"맞춘 직선 y={slope:.2f}x+{intercept:.2f}")
axes[0].vlines(x, y_fit, y, color="gray", lw=0.8)   # 잔차를 선으로 표시
axes[0].set_ylabel("y")
axes[0].set_title("최소제곱 직선 맞춤과 잔차")
axes[0].legend()

axes[1].axhline(0, color="red", lw=1)
axes[1].scatter(x, residual, color="darkorange")
axes[1].set_xlabel("x")
axes[1].set_ylabel("잔차")
axes[1].grid(alpha=0.3)

du.save_figure(fig, os.path.join(proj_dir, "linear_fit"))
plt.show()

**code 28.**

In [ ]:
from scipy.optimize import curve_fit

def decay(t, amplitude, tau, offset):
    """지수 감소 모형: 바닥값 위의 몫이 시간상수 tau마다 1/e(약 0.37)배로 줄어듦"""
    return amplitude * np.exp(-t / tau) + offset


t = np.linspace(0, 20, 40)
true_params = (100.0, 5.0, 10.0)                       # 진폭, 시간상수, 바닥값
signal = decay(t, *true_params) + rng.normal(0, 3.0, t.size)

params, covariance = curve_fit(decay, t, signal, p0=(80, 3, 5))
errors = np.sqrt(np.diag(covariance))                  # 대각성분의 제곱근 = 표준오차

for name, fit, err, true in zip(["진폭", "시간상수", "바닥값"],
                                params, errors, true_params):
    pad = " " * (10 - 2 * len(name))                    # 한글 한 글자는 두 칸 너비
    print(f"{name}{pad}: {fit:7.2f} +/- {err:5.2f}   (참값 {true})")

**code 29.**

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.scatter(t, signal, s=25, color="steelblue", label="관측값")
t_smooth = np.linspace(t.min(), t.max(), 300)
ax.plot(t_smooth, decay(t_smooth, *params), "r-", lw=2, label="맞춘 곡선")
ax.set_xlabel("시간")
ax.set_ylabel("신호 세기")
ax.set_title(f"지수 감소 모형 맞춤 (시간상수 {params[1]:.2f})")
ax.legend()
ax.grid(alpha=0.3)
du.save_figure(fig, os.path.join(proj_dir, "curve_fit"))
plt.show()

**code 30.**

In [ ]:
def r_squared(observed, predicted):
    """결정계수: 1에 가까울수록 모형이 데이터의 변동을 잘 설명한 것."""
    ss_residual = np.sum((observed - predicted) ** 2)
    ss_total = np.sum((observed - observed.mean()) ** 2)
    return 1 - ss_residual / ss_total


print("직선 맞춤 R^2      :", round(r_squared(y, y_fit), 4))
print("지수 모형 R^2      :", round(r_squared(signal, decay(t, *params)), 4))

print("\n다항식 차수별 R^2 (같은 데이터)")
for degree in (1, 3, 7, 15):
    coefficients = np.polyfit(x, y, degree)
    print(f"  {degree:2d}차 : {r_squared(y, np.polyval(coefficients, x)):.4f}")

**code 31.**

In [ ]:
x_dense = np.linspace(x.min() - 0.5, x.max() + 0.5, 400)   # 데이터 범위보다 조금 넓게

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.scatter(x, y, color="steelblue", zorder=3, label="관측값")
for degree, style in zip((1, 15), ("r-", "g--")):
    coefficients = np.polyfit(x, y, degree)
    ax.plot(x_dense, np.polyval(coefficients, x_dense), style, lw=1.8,
            label=f"{degree}차 다항식")
ax.set_ylim(y.min() - 5, y.max() + 5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("차수를 올리면 잘 맞는 것처럼 보이지만")
ax.legend()
ax.grid(alpha=0.3)
du.save_figure(fig, os.path.join(proj_dir, "overfitting"))
plt.show()